# NC3 — Tract-Level Mismatch Snapshot
## Neighborhood Indicators for Partner Discussion
**Date:** November 04, 2026  
**Prepared for:** Lansing Growth Fund  
**Focus:** Ranking ACS 2020–2024 tracts by poverty and median income as a starting point for neighborhood mismatch analysis

This notebook loads the tract extract already in `Data_Files/acs_tract_2020_2024.csv` and surfaces the highest-poverty and lowest-income tracts in the tri-county region. It is a lightweight starting point for the neighborhood mismatch research question; full spatial overlay with institutional sites is a stretch goal.

Student-heavy tracts (especially near MSU / East Lansing) can show elevated poverty and depressed median income independent of institutional job access.

In [ ]:
import pandas as pd
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', '{:,.0f}'.format)

# Adjust path if needed
tracts = pd.read_csv('../Data_Files/acs_tract_2020_2024.csv')
print(f'Loaded {len(tracts)} tracts')
print(tracts['county_name'].value_counts())

In [ ]:
# Poverty rate
tracts['poverty_rate'] = (tracts['below_poverty'] / tracts['poverty_universe'] * 100).round(1)
tracts['owner_occ_share'] = (tracts['owner_occupied'] / tracts['occupied_units'] * 100).round(1)

summary = tracts.groupby('county_name').agg(
    tracts=('tract', 'count'),
    median_income=('median_hh_income', 'median'),
    mean_poverty=('poverty_rate', 'mean')
).round(1)
print('County-level tract summary')
display(summary)

## Highest-Poverty Tracts (top 15)

In [ ]:
high_pov = tracts.nlargest(15, 'poverty_rate')[[
    'NAME', 'county_name', 'poverty_rate', 'median_hh_income', 'population', 'owner_occ_share'
]]
print('Top 15 tracts by poverty rate')
display(high_pov)

fig = px.bar(
    high_pov.head(10),
    x='poverty_rate',
    y='NAME',
    orientation='h',
    color='county_name',
    title='Top 10 Tracts by Poverty Rate (ACS 2020–2024)',
    labels={'poverty_rate': 'Poverty Rate (%)', 'NAME': ''}
)
fig.update_layout(template='plotly_white', height=480, title_x=0.5, yaxis={'categoryorder': 'total ascending'})
fig.show()

## Lowest Median Household Income Tracts (top 15)

In [ ]:
low_inc = tracts.nsmallest(15, 'median_hh_income')[[
    'NAME', 'county_name', 'median_hh_income', 'poverty_rate', 'population', 'owner_occ_share'
]]
print('15 lowest-income tracts')
display(low_inc)

fig = px.bar(
    low_inc.head(10),
    x='median_hh_income',
    y='NAME',
    orientation='h',
    color='county_name',
    title='10 Lowest Median Household Income Tracts (ACS 2020–2024)',
    labels={'median_hh_income': 'Median HH Income ($)', 'NAME': ''}
)
fig.update_layout(template='plotly_white', height=480, title_x=0.5, yaxis={'categoryorder': 'total ascending'})
fig.show()

## Notes for Partner Discussion

- High-poverty and low-income tracts cluster in Ingham County (Lansing / East Lansing area).
- Several of the highest-poverty tracts also show low owner-occupancy, consistent with rental and student housing patterns.
- Next step: join tract centroids or boundaries to institutional site locations (MSU campus, GM Grand River, Delta Township Assembly) and compute distance or containment flags.
- Opportunity Atlas mobility measures (see `Data_Files/next_sources.md`) would add a longer-term outcome layer once joined on GEOID.

---
*Notebook prepared for the NC3 community partner meeting. Load `Data_Files/acs_tract_2020_2024.csv` directly for production refresh.*